# Frozen-checkpoint test evaluation and visualization

This notebook evaluates only a checkpoint selected from train/validation. It performs no threshold tuning, hyperparameter search, early-stopping decision, or model comparison. Predictions use two-logit `argmax`.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import numpy as np
import pandas as pd
import torch

from reservoir_segmentation.geometry import shoreline_boundary
from reservoir_segmentation.training.data import (
    ChannelStatistics, build_dataloader, build_split_datasets,
)
from reservoir_segmentation.training.engine import evaluate, model_from_checkpoint


In [ ]:
REPOSITORY_ROOT = next((path for path in (Path.cwd().resolve(), *Path.cwd().resolve().parents) if (path / 'pyproject.toml').is_file()), None)
if REPOSITORY_ROOT is None:
    raise FileNotFoundError('Could not locate the repository root. Start Jupyter from the repository or its notebooks directory.')
GEOMETRY_ROOT = REPOSITORY_ROOT / 'data/processed/riverscope/geometry-v2'
SOURCE_PATH = REPOSITORY_ROOT / 'data/raw/RiverScope.zip'
CHECKPOINT_PATH = REPOSITORY_ROOT / 'artifacts/experiments/final_train_validation/best_checkpoint.pt'
OUTPUT_ROOT = REPOSITORY_ROOT / 'artifacts/evaluations/final_test'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

model, config, checkpoint = model_from_checkpoint(CHECKPOINT_PATH, device=DEVICE)
statistics = ChannelStatistics.from_dict(checkpoint['channel_statistics'])
datasets = build_split_datasets(
    GEOMETRY_ROOT, SOURCE_PATH, statistics=statistics, include_test=True,
)
test_loader = build_dataloader(
    datasets['test'], batch_size=config.optimization.batch_size, shuffle=False,
    seed=config.runtime.seed, num_workers=config.runtime.num_workers,
)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)


## One final test pass

The result table preserves undefined ASSD/HD95 values and boundary-case labels. `selection_assd_px` remains a transparent diagnostic penalty for one-empty-boundary failures; it does not retroactively select this checkpoint.

In [ ]:
test_result = evaluate(model, test_loader, config.loss, device=DEVICE, progress_label='test')
tile_table = pd.DataFrame(test_result['tiles']).sort_values('selection_assd_px')
tile_table['visualization_path'] = [
    str(OUTPUT_ROOT / 'tiles' / f'{tile_id}.png') for tile_id in tile_table['tile_id']
]
tile_table.to_csv(OUTPUT_ROOT / 'test_tile_metrics.csv', index=False)
test_summary = {key: value for key, value in test_result.items() if key != 'tiles'}
test_summary


## Reproducible representative cases

The first selection is random with a fixed seed; the remaining selections are best and worst tiles by the predeclared test diagnostic metric. Invalid pixels appear in gray and are excluded from all overlays and errors.

In [ ]:
rng = np.random.default_rng(44)
random_path = rng.choice(tile_table['image_path'].to_numpy())
best_path = tile_table.iloc[0]['image_path']
worst_path = tile_table.iloc[-1]['image_path']
sample_index = {row['normalized_planetscope_path']: index for index, row in enumerate(datasets['test'].rows)}

def display_rgb(chw):
    rgb = np.moveaxis(chw[:3], 0, -1)
    low, high = np.quantile(rgb, (0.02, 0.98))
    return np.clip((rgb - low) / max(high - low, 1e-6), 0, 1)

def render_tile(image_path, label, show=True):
    sample = datasets['test'][sample_index[image_path]]
    normalized = sample['image'].numpy()
    mean = np.asarray(statistics.mean)[:, None, None]
    std = np.asarray(statistics.std)[:, None, None]
    image = normalized * std + mean
    target = sample['target'].numpy().astype(bool)
    valid = sample['valid_mask'].numpy().astype(bool)
    rgb = display_rgb(image); rgb[~valid] = 0.5
    reference_boundary = sample['boundary'].numpy().astype(bool)
    with torch.no_grad():
        probabilities = model(sample['image'].unsqueeze(0).to(DEVICE)).softmax(dim=1)[0, 1].cpu().numpy()
    prediction = model.predict_labels(sample['image'].unsqueeze(0).to(DEVICE))[0].cpu().numpy().astype(bool)
    predicted_boundary = shoreline_boundary(prediction, valid, connectivity=8)
    errors = np.full(target.shape, np.nan)
    errors[valid & ~target & ~prediction] = 0
    errors[valid & target & prediction] = 1
    errors[valid & target & ~prediction] = 2
    errors[valid & ~target & prediction] = 3
    fig, axes = plt.subplots(2, 4, figsize=(18, 9))
    axes = axes.ravel()
    axes[0].imshow(rgb); axes[0].set_title('RGB composite; gray invalid')
    axes[1].imshow(np.ma.masked_where(~valid, target), cmap='Blues', vmin=0, vmax=1); axes[1].set_title('Ground truth')
    axes[2].imshow(np.ma.masked_where(~valid, prediction), cmap='Blues', vmin=0, vmax=1); axes[2].set_title('Argmax prediction')
    axes[3].imshow(np.ma.masked_where(~valid, probabilities), cmap='viridis', vmin=0, vmax=1); axes[3].set_title('Water probability')
    axes[4].imshow(reference_boundary, cmap='magma'); axes[4].set_title('Reference shoreline')
    axes[5].imshow(predicted_boundary, cmap='magma'); axes[5].set_title('Predicted shoreline')
    axes[6].imshow(rgb); axes[6].contour(reference_boundary, colors='lime', linewidths=0.7); axes[6].contour(predicted_boundary, colors='red', linewidths=0.7); axes[6].set_title('Reference green / prediction red')
    axes[7].imshow(errors, cmap=ListedColormap(['black', 'lime', 'orange', 'red']), vmin=0, vmax=3); axes[7].imshow(np.where(valid, np.nan, 1), cmap=ListedColormap(['gray']), alpha=0.75); axes[7].set_title('TN / TP / FN / FP; gray invalid')
    for axis in axes: axis.set_axis_off()
    fig.suptitle(f'{label}: {image_path}', fontsize=9); fig.tight_layout()
    destination = OUTPUT_ROOT / f'{label}.png'
    destination.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(destination, dpi=160)
    if show: plt.show()
    else: plt.close(fig)

for label, path in [('random', random_path), ('best', best_path), ('worst', worst_path)]:
    render_tile(path, label)


In [ ]:
EXPORT_ALL_TILE_VISUALIZATIONS = True
if EXPORT_ALL_TILE_VISUALIZATIONS:
    for row in tile_table.itertuples(index=False):
        render_tile(row.image_path, f'tiles/{row.tile_id}', show=False)
